# 6.5 · HDBSCAN / Hierarchical DBSCAN

> **课程定位 / Where this fits**
> 6.4 的 DBSCAN 卡在**单一全局 eps**——无法同时处理不同密度的簇。HDBSCAN(Hierarchical DBSCAN)把 DBSCAN 在**所有密度尺度**上层次化, 再用簇的**稳定性**自动抽取最持久的簇。几乎不用手调 eps, 自动定簇数、自动识别噪声、适应不同密度。是现代密度聚类的首选。
> HDBSCAN runs DBSCAN across all density scales, then extracts the most stable clusters — no global eps to tune, adapts to varying density.

> 💡 **面试相关 / Interview-relevant**
> - "HDBSCAN 相比 DBSCAN 解决了什么" ★★★★★（不同密度 + 不调eps）
> - "互达距离 / 簇稳定性 是什么" ★★★★
> - "min_cluster_size 的作用" ★★★★
> - "软聚类 / 概率成员" ★★★

---

## 学习目标 / Learning Objectives
1. HDBSCAN 直觉: 跨所有 eps 的层次 + 稳定性抽取。
2. 互达距离与最小生成树(概念层面)。
3. `min_cluster_size` / `min_samples` 调参。
4. 在不同密度数据上完胜 DBSCAN。
5. 软成员概率与异常分数。

## 目录 / TOC
1. [从 DBSCAN 到 HDBSCAN ⭐](#1)
2. [🗺️ 数据: 不同密度地理点](#2)
3. [HDBSCAN vs DBSCAN ⭐](#3)
4. [min_cluster_size + 概率成员](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 从 DBSCAN 到 HDBSCAN ⭐ / From DBSCAN to HDBSCAN

DBSCAN 固定一个 eps, 相当于在密度图上"切一个高度"。HDBSCAN 的思路: **不切固定高度, 而是看所有高度**。

1. **互达距离(mutual reachability)**: 重定义点距 $d_{mreach}(a,b)=\max(\text{core}_k(a), \text{core}_k(b), d(a,b))$, 其中 core 是到第 k 近邻的距离。它**拉开稀疏区**的点, 让密度差异显式化。
2. **最小生成树 → 层次**: 在互达距离上建 MST, 逐步提高距离阈值"砍边", 得到一棵簇的合并/分裂层次(像 6.3 树状图, 但基于密度)。
3. **稳定性抽取**: 一个簇在很宽的密度范围内都存在 → **稳定**; 一闪而过 → 不稳定。HDBSCAN 选出稳定性最大的一组簇作为最终结果。

结果: **不同密度的簇各按自己的尺度被识别**, 无需统一 eps。


<a id="2"></a>
## 2. 数据: 不同密度地理点 / Geo Points with Varying Density

模拟地图上的兴趣点(POI): 市中心**密集**、郊区**稀疏**、外加随机噪声。这正是 DBSCAN 用单一 eps 会翻车、HDBSCAN 该发光的场景。
> 工具说明: sklearn ≥1.3 内置 `sklearn.cluster.HDBSCAN`, 无需第三方包。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import HDBSCAN, DBSCAN
sns.set_theme(style="whitegrid")

rng = np.random.default_rng(0)
downtown = rng.normal([0, 0], 0.25, (400, 2))      # 密集市中心
mall     = rng.normal([4, 1], 0.4, (150, 2))       # 中密度
suburb1  = rng.normal([2, 5], 1.1, (120, 2))       # 稀疏郊区
suburb2  = rng.normal([-3, 4], 1.0, (120, 2))      # 稀疏郊区
noise    = rng.uniform([-6,-3], [8,8], (80, 2))    # 随机噪声
X = np.vstack([downtown, mall, suburb1, suburb2, noise])
print(f"地理点: {X.shape} (1 密集 + 1 中 + 2 稀疏 簇 + 噪声)")

fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(X[:,0], X[:,1], s=10, alpha=0.5)
ax.set_title("地理 POI: 市中心密集 / 郊区稀疏 / 随机噪声(待聚类)")
ax.set_xlabel("经度(相对)"); ax.set_ylabel("纬度(相对)")
plt.tight_layout(); plt.show()


<a id="3"></a>
## 3. HDBSCAN vs DBSCAN ⭐ / Head-to-head


In [ ]:
def summarize(lab):
    n_clu = len(set(lab)) - (1 if -1 in lab else 0)
    return n_clu, (lab == -1).sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
# DBSCAN 两个不同 eps
for ax, eps in zip(axes[:2], [0.3, 0.9]):
    lab = DBSCAN(eps=eps, min_samples=8).fit_predict(X)
    nc, nn = summarize(lab); m = lab==-1
    ax.scatter(X[~m,0], X[~m,1], c=lab[~m], cmap="tab10", s=12)
    ax.scatter(X[m,0], X[m,1], c="lightgray", marker="x", s=15)
    ax.set_title(f"DBSCAN eps={eps}: {nc}簇, {nn}噪声")

# HDBSCAN 自动
hlab = HDBSCAN(min_cluster_size=30).fit_predict(X)
nc, nn = summarize(hlab); m = hlab==-1
axes[2].scatter(X[~m,0], X[~m,1], c=hlab[~m], cmap="tab10", s=12)
axes[2].scatter(X[m,0], X[m,1], c="lightgray", marker="x", s=15)
axes[2].set_title(f"HDBSCAN(自动): {nc}簇, {nn}噪声")
plt.tight_layout(); plt.show()
print("DBSCAN eps=0.3: 稀疏郊区被打散成噪声; eps=0.9: 密集中心和邻簇被合并")
print("HDBSCAN: 不调 eps, 同时正确识别密集与稀疏簇 + 噪声 → 适应不同密度")


<a id="4"></a>
## 4. min_cluster_size + 概率成员 / Params & Soft Membership

- **min_cluster_size**: 最小簇规模——小于它的密度团被当噪声。HDBSCAN 最主要的旋钮, 比 eps 直观得多(直接是"多大才算一个簇")。
- **min_samples**: 控制保守程度(越大越多点判为噪声)。
- **软成员**: HDBSCAN 给每点一个 `probabilities_`(属于所分簇的强度), 边缘点概率低——可用于异常评分。


In [ ]:
h = HDBSCAN(min_cluster_size=30, store_centers="medoid").fit(X)
print(f"簇数: {len(set(h.labels_))-(1 if -1 in h.labels_ else 0)}, 噪声: {(h.labels_==-1).sum()}")

fig, ax = plt.subplots(figsize=(7,5))
sc_ = ax.scatter(X[:,0], X[:,1], c=h.probabilities_, cmap="viridis", s=14)
plt.colorbar(sc_, label="簇成员概率")
ax.set_title("HDBSCAN 软成员: 簇核心概率高, 边缘/噪声概率低(可做异常评分)")
plt.tight_layout(); plt.show()

# min_cluster_size 影响 / effect
for mcs in [15, 30, 60, 120]:
    lab = HDBSCAN(min_cluster_size=mcs).fit_predict(X)
    nc = len(set(lab))-(1 if -1 in lab else 0)
    print(f"min_cluster_size={mcs:>3}: {nc} 簇, {(lab==-1).sum()} 噪声")
print("min_cluster_size 越大 → 只保留更大的簇, 小团并入噪声")


<a id="5"></a>
## 5. 小结 / Summary

```
HDBSCAN = 跨所有密度尺度的 DBSCAN 层次 + 稳定性抽取
互达距离: max(core_k(a), core_k(b), d(a,b)) → 显式化密度差异
MST→层次→选最稳定簇; 不同密度簇各按自己尺度识别, 无需统一 eps
主旋钮 min_cluster_size(多大才算簇, 比 eps 直观); 软成员概率可做异常评分
适应不同密度 + 自动定簇数 + 自动噪声 → 现代密度聚类首选
```

### 💡 面试速查
1. **解决 DBSCAN 的不同密度+调eps 痛点**: 层次化所有密度尺度 + 稳定性抽取
2. **互达距离**拉开稀疏区; MST 建层次; 选最稳定的簇
3. **min_cluster_size** 是主旋钮(比 eps 直观), min_samples 控保守度
4. **软成员概率**: 边缘点低概率, 可用于异常检测
5. sklearn≥1.3 内置 `cluster.HDBSCAN`

### 下一节
**6.6 高斯混合 GMM**——前面都是硬分配(一个点属一个簇)。GMM 用 EM 算法做**软分配**(概率属于各簇), 还能建模椭圆形簇, 是生成式聚类。
